# Day 2: Exploratory Data Analysis & Visualization
### Course: Data Analytics Complete

Welcome to the Day 2 hands-on laboratory! Today, we will transition from simple data collection to **Data Understanding**. We will cover:
1. **Exploratory Data Analysis (EDA)**: Understanding the 'shape' and 'health' of our data.
2. **Visualization Fundamentals**: Choosing the right chart for the right story.
3. **Advanced Visualization**: Building professional dashboards.

---

## 0. Setup & Data Creation
In a real-world scenario, you would load data from a database or a CSV. For this lab, we will generate the `students_data.csv` first so everything is self-contained.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from io import StringIO

# Set style for better visuals
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Create the dataset
csv_data = """name,age,gender,city,study_hours,attendance,marks
Arjun,20,Male,Bangalore,7,92,88
Priha,21,Female,Chennai,,85,78
Vikram,19,Male,Hyderabad,8,98,95
Sneha,22,Female,Bangalore,4,,68
Rahul,20,Male,Chennai,5,80,72
Ananya,21,Female,Hyderabad,6,88,84
Rohan,23,Male,Bangalore,9,95,92
Divya,20,Female,Chennai,3,70,60
Karan,21,Male,Hyderabad,7,90,82
Sonia,19,Female,Bangalore,6,85,80
"""

df = pd.read_csv(StringIO(csv_data))
df.to_csv("students_data.csv", index=False)
print("✅ Dataset 'students_data.csv' created and loaded!")
df.head()
df.info()

## Section 1: Data Understanding & Cleaning
Before analysis, we must ensure our data is clean. "Garbage in, Garbage out"!

In [ ]:
print(f"\n[1] Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Check for missing values
print("\n[2] Missing Values per Column:")
print(df.isnull().sum())

# Handle Missing Values with the MEAN
df["study_hours"] = df["study_hours"].fillna(df["study_hours"].mean())
df["attendance"]  = df["attendance"].fillna(df["attendance"].mean())
df = df.drop_duplicates().reset_index(drop=True)

print("\n✅ Data Cleaned! No nulls or duplicates remain.")
df.describe().round(2)

### Deep Dive: Individual Metrics\nSometimes we need specific numbers rather than the whole table.

In [ ]:
print(f"Average Marks: {df[\"marks\"].mean():.2f}")\nprint(f"Median Marks:  {df[\"marks\"].median()}")\nprint(f"Standard Dev:  {df[\"marks\"].std():.2f}")\nprint(f"Min Marks:     {df[\"marks\"].min()}")\nprint(f"Max Marks:     {df[\"marks\"].max()}")\nprint(f"25th Percentile: {df[\"marks\"].quantile(0.25)}")\nprint(f"75th Percentile: {df[\"marks\"].quantile(0.75)}")\n\nprint("\\nCity Frequency:")\nprint(df[\"city\"].value_counts())

### Data Bucketing (Grouping)\nUseful for categorizing continuous data into ranges.

In [ ]:
df[\"performance_cat\"] = pd.cut(df[\"marks\"], bins=[0, 70, 85, 100], labels=[\"Below Average\", \"Average\", \"Excellent\"])\nprint("Performance Categories:")\nprint(df[\"performance_cat\"].value_counts())

## Section 2: Statistical EDA
Statistical EDA helps us find patterns and outliers numerically.

In [ ]:
print("=== CORRELATION ANALYSIS ===")
corr_matrix = df[["study_hours", "attendance", "marks"]].corr()
print(corr_matrix.round(2))

print("\n=== OUTLIER DETECTION (IQR METHOD) ===")
for col in ["marks", "study_hours"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col:12} | Fence: [{lower:.1f}, {upper:.1f}] | Outliers: {len(outliers)}")

## Section 3: Visualization Fundamentals
Now we use charts to tell the story of our students.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Bar Chart: Avg Marks by City
avg_city = df.groupby("city")["marks"].mean().sort_values(ascending=False)
sns.barplot(x=avg_city.index, y=avg_city.values, ax=axes[0, 0], palette="viridis")
axes[0, 0].set_title("Avg Marks by City", fontweight='bold')
axes[0, 0].set_ylim(0, 100)

# 2. Line Chart: Trend (Simulated Monthly Data)
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
marks_trend = [70, 72, 75, 78, 82, 85]
axes[0, 1].plot(months, marks_trend, marker='o', linewidth=2, color='red')
axes[0, 1].set_title("Monthly Learning Trend", fontweight='bold')
axes[0, 1].set_ylim(60, 90)

# 3. Histogram: Marks Distribution
sns.histplot(df["marks"], bins=6, kde=True, ax=axes[1, 0], color="skyblue")
axes[1, 0].set_title("Distribution of Student Marks", fontweight='bold')

# 4. Boxplot: Marks by Gender
sns.boxplot(data=df, x="gender", y="marks", ax=axes[1, 1], palette="Pastel1")
axes[1, 1].set_title("Performance Spread by Gender", fontweight='bold')
axes[1, 1].set_ylim(0, 110)

plt.tight_layout()
plt.show()

## Section 4: Advanced Visualization
Going beyond basic charts: Heatmaps and Distribution shapes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Correlation Heatmap
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", ax=axes[0])
axes[0].set_title("Correlation Heatmap", fontweight='bold')

# 2. Violin Plot (Density + Boxplot)
sns.violinplot(data=df, x="city", y="marks", inner="quartile", ax=axes[1], palette="Set3")
axes[1].set_title("Violin Plot: Marks Density by City", fontweight='bold')

plt.show()

## Section 5: The Analytics Dashboard (Capstone)
We will now combine everything into a professional 6-panel dashboard using `gridspec`.

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle("📊 Student Performance Analytics Dashboard", fontsize=20, fontweight='bold', y=1.02)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

# Panel 1: Barchart
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x=df["city"], y=df["marks"], ax=ax1, errorbar=None, palette="magma")
ax1.set_title("City-wise Performance", fontweight='bold')

# Panel 2: Scatter Plot
ax2 = fig.add_subplot(gs[0, 1])
sns.scatterplot(data=df, x="study_hours", y="marks", hue="city", size="attendance", sizes=(100, 400), ax=ax2)
ax2.set_title("Study Hours vs Marks", fontweight='bold')

# Panel 3: KPI Metrics
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('off')
kpi_text = (
    f"Total Students: {len(df)}\n"
    f"Average Marks: {df['marks'].mean():.1f}\n"
    f"Avg Attendance: {df['attendance'].mean():.1f}%\n"
    f"Top Student: {df.loc[df['marks'].idxmax(), 'name']}"
)
ax3.text(0.5, 0.5, kpi_text, ha='center', va='center', fontsize=15, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax3.set_title("Key Performance Indicators", fontweight='bold')

# Panel 4: Pie Chart
ax4 = fig.add_subplot(gs[1, 0])
city_counts = df["city"].value_counts()
ax4.pie(city_counts, labels=city_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("Set2"))
ax4.set_title("Student Distribution by City", fontweight='bold')

# Panel 5: Distribution (KDE)
ax5 = fig.add_subplot(gs[1, 1])
sns.kdeplot(data=df, x="marks", hue="gender", fill=True, ax=ax5)
ax5.set_title("Marks Distribution by Gender", fontweight='bold')

# Panel 6: Insight Box
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
insights = (
    "1. Strong Correlation (0.96) between\n   Study Hours and Marks.\n"
    "2. Bangalore is the leading city.\n"
    "3. Students with <75% attendance\n   tend to score lower."
)
ax6.text(0, 0.5, insights, ha='left', va='center', fontsize=12, 
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax6.set_title("Actionable Insights", fontweight='bold')

plt.show()

## Final Takeaways & Ethics
1. **Ethical Visualization**: Always start your Y-axis at 0 for bar charts to avoid misleading the audience.
2. **Simplification**: Don't clutter charts; use color and labels only when they add value.
3. **Storytelling**: Data represents people. Beyond the numbers, look for the 'Why' (e.g., Why do Hyderabad students study more?).

---